# CineMatch Pro — Kaggle Training Notebook

Is notebook ko **upar se neeche, cell by cell (ya Run All)** chalana hai. Har code cell ke top par `# CELL n` likha hai.

## Kaise kaam karta hai (2 alag cheezein)
| Cheez | Kya hai | Kaise add hoti hai |
|---|---|---|
| **Ye notebook** (`Kaggle_Training.ipynb`) | Instructions/commands | Kaggle → *New Notebook* → *File → Import Notebook* |
| **Project zip** (`cinematch-pro-complete.zip`) + **datasets** | Code aur data jo notebook use karta hai | Right panel → *Input → Add Input* |

## Pehle ye setup (ek baar)
1. **Settings (right panel):** Accelerator → **GPU T4 x2** · Internet → **On** (phone-verified account chahiye).
2. **Add Input** se ye data jodo (Kaggle ke public datasets search karke, ya apna uploaded subset dataset):
   - MovieLens 25M — `ratings.csv`, `movies.csv`, `links.csv`
   - *IMDB Dataset of 50K Movie Reviews* — `IMDB Dataset.csv`
   - *IMDB TMDB Movie Metadata Big Dataset (1M)* — one big csv
   - *Movie Genre from its Poster* — `MovieGenre.csv`
   - **Project:** `cinematch-pro-complete.zip` ko apne account me ek *private Dataset* bana ke jodo (Datasets → New Dataset → zip upload). Kaggle zip ko khud extract kar deta hai; CELL 5 dono tarah handle karta hai.
3. **CELL 3** me `RUN_MODE` chuno: pehli baar `"QUICK"` (chhota run, sirf check karne ke liye), phir `"FULL"`.

> Lamba run ho to *Save Version → Save & Run All (Commit)* use karo — notebook background me chalta hai, tab band kar sakte ho. Koi cell input nahi maangta.

---
**Updated version:** FULL mode + credits fix (CELL 8b) + `--nrows {N_RATINGS}`.


In [1]:
# CELL 1 — Install the packages Kaggle does not ship (scikit-surprise compiles from source: allow a few minutes)
!pip install -q scikit-surprise faiss-cpu chromadb pyarrow python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 88.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━

Agar CELL 2 me import error aaye (jaise *"numpy.dtype size changed"*): **Run → Restart session**, phir CELL 1 se dobara chalao.
`scikit-surprise` naye Kaggle image pe sabse pehle fail hone wala package hai.

In [2]:
# CELL 2 — Check the GPU and library versions
import importlib, platform
import torch

print("Python", platform.python_version())
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for pkg in ["numpy", "pandas", "sklearn", "surprise", "faiss", "chromadb", "transformers", "sentence_transformers"]:
    try:
        module = importlib.import_module(pkg)
        print(f"{pkg:22s}", getattr(module, "__version__", "?"))
    except Exception as exc:  # noqa: BLE001
        print(f"{pkg:22s} FAILED -> {type(exc).__name__}: {exc}")

assert torch.cuda.is_available(), "GPU is OFF: Settings -> Accelerator -> GPU T4 x2, then restart the session."


Python 3.13.15
CUDA available: True | GPUs: 2
numpy                  2.1.3
pandas                 2.3.3
sklearn                1.6.1
surprise               1.1.5
faiss                  1.15.1
chromadb               1.5.9
transformers           5.16.1
sentence_transformers  5.7.0


In [3]:
# CELL 3 — Choose how big this run is. Change ONLY this line.
RUN_MODE = "FULL"        # "QUICK" = small run to prove everything works | "FULL" = final training run

SETTINGS = {
    #          ratings used by SVD | IMDB reviews for RoBERTa | posters to index
    "QUICK": dict(n_ratings=1_000_000, sentiment_sample=5_000, poster_limit=2_000),
    "FULL":  dict(n_ratings=5_000_000, sentiment_sample=None,  poster_limit=None),
}
assert RUN_MODE in SETTINGS, "RUN_MODE must be 'QUICK' or 'FULL'"
N_RATINGS = SETTINGS[RUN_MODE]["n_ratings"]
SENT_ARGS = f"--sample {SETTINGS[RUN_MODE]['sentiment_sample']}" if SETTINGS[RUN_MODE]["sentiment_sample"] else ""
POSTER_ARGS = f"--limit {SETTINGS[RUN_MODE]['poster_limit']}" if SETTINGS[RUN_MODE]["poster_limit"] else ""
print(f"RUN_MODE={RUN_MODE}: ratings={N_RATINGS:,} | sentiment sample={SETTINGS[RUN_MODE]['sentiment_sample'] or 'ALL 50k'} "
      f"| posters={SETTINGS[RUN_MODE]['poster_limit'] or 'ALL'}")
assert RUN_MODE == "FULL", "Still in QUICK mode!"


RUN_MODE=FULL: ratings=5,000,000 | sentiment sample=ALL 50k | posters=ALL


In [4]:
# CELL 4 — Find your data files under /kaggle/input (names are matched, folders don't matter)
from pathlib import Path

INPUT = Path("/kaggle/input")
all_files = [p for p in INPUT.rglob("*") if p.is_file()]
lname = lambda p: p.name.lower()


def find(predicate, what, prefer=None):
    hits = [p for p in all_files if predicate(p)]
    if not hits:
        seen = [str(p.relative_to(INPUT)) for p in all_files[:40]]
        raise FileNotFoundError(f"Could not find {what} under /kaggle/input. First files seen: {seen}")
    if prefer:  # several datasets can contain a movies.csv; prefer the MovieLens 25M one
        hits = [p for p in hits if prefer in str(p).lower()] or hits
    if len(hits) > 1:
        print(f"NOTE: {len(hits)} candidates for {what}; using {hits[0]}")
    return hits[0]


RATINGS = find(lambda p: lname(p) == "ratings.csv", "MovieLens ratings.csv", prefer="25m")
MOVIES = find(lambda p: lname(p) == "movies.csv", "MovieLens movies.csv", prefer="25m")
LINKS = find(lambda p: lname(p) == "links.csv", "MovieLens links.csv", prefer="25m")
IMDB = find(lambda p: "imdb dataset" in lname(p) and lname(p).endswith(".csv") and "tmdb" not in lname(p),
            "IMDB reviews 'IMDB Dataset.csv'")
BIG = find(lambda p: "big dataset" in lname(p) and lname(p).endswith(".csv"), "the 'Big Dataset (1M)' TMDB csv")
GENRE = find(lambda p: lname(p) == "moviegenre.csv", "MovieGenre.csv")

for label, path in [("ratings", RATINGS), ("movies", MOVIES), ("links", LINKS), ("imdb reviews", IMDB),
                    ("tmdb big", BIG), ("poster genre", GENRE)]:
    print(f"{label:14s} {path}  ({path.stat().st_size / 1e6:,.0f} MB)")

ratings        /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/ratings.csv  (678 MB)
movies         /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/movies.csv  (3 MB)
links          /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/links.csv  (1 MB)
imdb reviews   /kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv  (66 MB)
tmdb big       /kaggle/input/datasets/shubhamchandra235/imdb-and-tmdb-movie-metadata-big-dataset-1m/IMDB TMDB Movie Metadata Big Dataset (1M).csv  (996 MB)
poster genre   /kaggle/input/datasets/neha1703/movie-genre-from-its-poster/MovieGenre.csv  (9 MB)


In [5]:
# CELL 5 — Unpack the project code into /kaggle/working/cinematch-pro
import os, shutil, sys, zipfile

WORK = Path("/kaggle/working")
PROJECT = WORK / "cinematch-pro"

if not (PROJECT / "src" / "config.py").exists():
    # Case A: Kaggle already extracted the project into the dataset.
    extracted = [p for p in all_files if p.name == "config.py" and p.parent.name == "src"]
    if extracted:
        shutil.copytree(extracted[0].parent.parent, PROJECT, dirs_exist_ok=True)
    else:
        # Case B: the zip is still a zip.
        zips = [p for p in all_files if p.suffix.lower() == ".zip" and "cinematch" in lname(p)]
        assert zips, "Project not found: add the dataset that contains cinematch-pro-complete.zip (or the folder)."
        with zipfile.ZipFile(zips[0]) as archive:
            archive.extractall(PROJECT)
        # A zip made from the folder itself nests everything one level down.
        if not (PROJECT / "src").exists():
            nested = next(p.parent for p in PROJECT.rglob("config.py") if p.parent.name == "src").parent
            for child in nested.iterdir():
                shutil.move(str(child), PROJECT / child.name)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from src.config import BASE_DIR, MODEL_CARD_PATH  # noqa: E402

print("Project root:", BASE_DIR)
assert (BASE_DIR / "scripts" / "build_cf_model.py").exists(), "scripts/ missing - wrong project folder?"


Project root: /kaggle/working/cinematch-pro


In [6]:
# CELL 6 — Fast sanity check of the code in THIS environment (a few seconds)
!python -m pytest tests/test_preprocessing.py tests/test_poster_utils.py tests/test_model_card.py \
    tests/test_content_filter.py tests/test_data_loader.py tests/test_cache_utils.py -q -p no:cacheprovider

.............................................................            [100%]
61 passed in 2.56s


## Stage 1 · Data merge → `merged_movies.parquet`
1M-row TMDB file ko MovieLens ke titles tak chhota karta hai, genres/cast/keywords parse karta hai, adult titles hatata hai,
har title+year ke liye sabse zyada votes wali row rakhta hai, aur har MovieLens `movieId` se jodta hai.

In [7]:
# CELL 7 — Preprocessing
!python -m src.preprocessing --tmdb-full-path "{BIG}" --ratings-path "{RATINGS}" --movies-path "{MOVIES}" --nrows {N_RATINGS}

2026-10-17 06:11:29 | INFO     | cinematch | Loading MovieLens ratings from /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/ratings.csv (nrows=5000000)
2026-10-17 06:11:32 | INFO     | cinematch | Loaded 5000000 ratings from 32564 unique users.
2026-10-17 06:11:32 | INFO     | cinematch | Loading MovieLens movies metadata from /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/movies.csv
2026-10-17 06:11:32 | INFO     | cinematch | Loaded metadata for 62423 movies.
2026-10-17 06:11:32 | INFO     | cinematch | Loading full TMDB metadata from /kaggle/input/datasets/shubhamchandra235/imdb-and-tmdb-movie-metadata-big-dataset-1m/IMDB TMDB Movie Metadata Big Dataset (1M).csv
2026-10-17 06:11:49 | INFO     | cinematch | Loaded full TMDB metadata for 1072255 movies.
2026-10-17 06:11:53 | INFO     | cinematch | Restricted TMDB metadata to MovieLens title/year matches: 1072255 -> 55087 rows.
2026-10-17 06:11:53 | INFO     | cinematch | Dropping 134 adult titles.
2026-10-17 06:11:58 | INFO    

In [8]:
# CELL 8 — Look at the merged result (note the TMDB match rate!)
import pandas as pd

movies = pd.read_parquet(PROJECT / "data/processed/merged_movies.parquet")
match_rate = movies["id"].notna().mean() if "id" in movies.columns else 0.0
print(f"{len(movies):,} MovieLens movies | unique movieId: {movies['movieId'].is_unique} | TMDB match rate: {match_rate:.1%}")
if match_rate < 0.4:
    print("WARNING: low TMDB match rate - note this number, recommendation quality depends on it.")
assert movies["movieId"].is_unique
display(movies[["movieId", "clean_title", "year", "director", "music_director"]].dropna(subset=["director"]).head(8))

62,423 MovieLens movies | unique movieId: True | TMDB match rate: 85.2%


,movieId,clean_title,year,director,music_director
0,1,Toy Story,1995,John Lasseter,Tyler Bates
1,2,Jumanji,1995,Sam Mendes,Norman Spencer
2,3,Grumpier Old Men,1995,Charles Crichton,John Du Prez
3,4,Waiting to Exhale,1995,Jaume Balagueró,Lucas Vidal
4,5,Father of the Bride Part II,1995,Xavier Gens,Geoff Zanelli
5,6,Heat,1995,Kenneth Branagh,
6,7,Sabrina,1995,Billy Wilder,
7,8,Tom and Huck,1995,Jay Curtis Miller,


## Stage 1b · Credits fix (TMDB API)
Big dataset ke `Director` / `Cast_list` / `Music_Composer` columns galat rows se jude hain (Heat → Kenneth Branagh). Is stage mein top 20,000 movies ke credits TMDB API se dobara aate hain (~11 min). **Pehle ek baar:** Kaggle → Add-ons → Secrets → `TMDB_API_KEY` add karo aur is notebook se attach karo.

In [9]:
%%writefile /kaggle/working/cinematch-pro/scripts/enrich_credits.py
"""Replace unreliable director/cast/composer/keywords with real TMDB API data.

Why this exists
---------------
The "IMDB & TMDB Movie Metadata Big Dataset (1M)" export has per-movie credit
columns (``Director``, ``Cast_list``, ``Music_Composer``) that are misaligned
with the movie rows (e.g. *Heat* (1995) lists Kenneth Branagh as director and
*Jumanji* lists Norman Spencer as composer). Titles, years, genres and
overviews are fine, but credits are not. This script re-fetches credits for
the most-voted movies straight from the TMDB API and blanks the credits of
movies it did not fetch (an honest "unknown" is better than a wrong name).

It then rebuilds ``combined_features`` so every downstream model (TF-IDF,
SBERT/ChromaDB) is trained on clean text. Run it AFTER ``src.preprocessing``
and BEFORE building the content filter / semantic index.

Usage:
    python -m scripts.enrich_credits --top 20000
    python -m scripts.enrich_credits --top 0      # every matched movie

The run is resumable: results are cached in ``credits_cache.jsonl``.
"""

from __future__ import annotations

import argparse
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import pandas as pd
import requests

from src.config import MERGED_MOVIES_PARQUET, TMDB_BASE_URL, logger

CACHE_PATH = MERGED_MOVIES_PARQUET.parent / "credits_cache.jsonl"
MAX_RETRIES = 4


class RateLimiter:
    """Thread-safe limiter that spaces requests to at most ``rate`` per second."""

    def __init__(self, rate: float) -> None:
        """Create a limiter.

        Args:
            rate: Maximum number of calls per second.
        """
        self._interval = 1.0 / rate
        self._lock = threading.Lock()
        self._next = time.monotonic()

    def wait(self) -> None:
        """Block until the next call is allowed."""
        with self._lock:
            now = time.monotonic()
            delay = self._next - now
            self._next = max(now, self._next) + self._interval
        if delay > 0:
            time.sleep(delay)


def parse_credits(payload: dict[str, Any]) -> dict[str, Any]:
    """Extract the fields CineMatch needs from a TMDB movie payload.

    Args:
        payload: JSON from ``/movie/{id}?append_to_response=credits,keywords``.

    Returns:
        dict[str, Any]: ``director``, ``cast_names`` (top 6), ``music_director``
        and ``keyword_names``.
    """
    credits = payload.get("credits") or {}
    crew = credits.get("crew") or []
    cast = sorted(credits.get("cast") or [], key=lambda c: c.get("order", 999))
    director = next((c["name"] for c in crew if c.get("job") == "Director"), "")
    composer = next(
        (c["name"] for c in crew if c.get("job") in {"Original Music Composer", "Composer", "Music"}),
        "",
    )
    kw_block = payload.get("keywords") or {}
    keywords = kw_block.get("keywords") or kw_block.get("results") or []
    return {
        "director": director,
        "cast_names": [c["name"] for c in cast[:6] if c.get("name")],
        "music_director": composer,
        "keyword_names": [k["name"] for k in keywords if k.get("name")][:15],
    }


def fetch_one(
    session: requests.Session, limiter: RateLimiter, api_key: str, tmdb_id: int
) -> dict[str, Any] | None:
    """Fetch and parse one movie, retrying on rate limits / transient errors.

    Args:
        session: Shared HTTP session.
        limiter: Shared rate limiter.
        api_key: TMDB v3 API key.
        tmdb_id: TMDB movie id.

    Returns:
        dict[str, Any] | None: Parsed credits (with ``id``), an empty-credits
        record for a 404, or ``None`` if every retry failed.
    """
    url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
    params = {"api_key": api_key, "append_to_response": "credits,keywords"}
    for attempt in range(MAX_RETRIES):
        limiter.wait()
        try:
            resp = session.get(url, params=params, timeout=15)
        except requests.RequestException as exc:
            logger.debug("TMDB request error for %s: %s", tmdb_id, exc)
            time.sleep(1 + attempt)
            continue
        if resp.status_code == 200:
            return {"id": tmdb_id, **parse_credits(resp.json())}
        if resp.status_code == 404:
            return {"id": tmdb_id, "director": "", "cast_names": [], "music_director": "", "keyword_names": []}
        if resp.status_code == 401:
            raise PermissionError("TMDB rejected the API key (HTTP 401).")
        if resp.status_code == 429:
            time.sleep(float(resp.headers.get("Retry-After", 2)))
            continue
        time.sleep(1 + attempt)
    return None


def load_cache() -> dict[int, dict[str, Any]]:
    """Load previously fetched credits.

    Returns:
        dict[int, dict[str, Any]]: Mapping of TMDB id to parsed credits.
    """
    cache: dict[int, dict[str, Any]] = {}
    if CACHE_PATH.exists():
        for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
            try:
                rec = json.loads(line)
                cache[int(rec["id"])] = rec
            except (ValueError, KeyError):
                continue
    return cache


def rebuild_features(row: pd.Series) -> str:
    """Rebuild the TF-IDF/SBERT text for one movie from clean fields.

    Args:
        row: A merged-movies row.

    Returns:
        str: Combined text (overview + genres + keywords + cast + director),
        falling back to MovieLens genres when everything else is empty.
    """
    parts: list[str] = []
    overview = row.get("overview")
    if isinstance(overview, str):
        parts.append(overview)
    for col in ("genre_names", "keyword_names", "cast_names"):
        val = row.get(col)
        if val is not None and not isinstance(val, str):
            parts.append(" ".join(str(v) for v in list(val)))
    director = row.get("director")
    if isinstance(director, str):
        parts.append(director)
    text = " ".join(p for p in parts if p).strip()
    if not text:
        text = str(row.get("genres", "")).replace("|", " ").strip()
    return text


def apply_credits(df: pd.DataFrame, cache: dict[int, dict[str, Any]]) -> pd.DataFrame:
    """Overwrite credit columns from the cache and blank everything unfetched.

    Args:
        df: Merged movies frame (needs an ``id`` column holding TMDB ids).
        cache: Fetched credits keyed by TMDB id.

    Returns:
        pd.DataFrame: Copy with clean ``director``, ``cast_names``,
        ``music_director``, ``keyword_names`` and rebuilt ``combined_features``.
    """
    out = df.copy()
    ids = pd.to_numeric(out["id"], errors="coerce")

    def pick(tmdb_id: Any, field: str, empty: Any) -> Any:
        if pd.isna(tmdb_id):
            return empty
        rec = cache.get(int(tmdb_id))
        return rec[field] if rec else empty

    out["director"] = [pick(i, "director", "") for i in ids]
    out["music_director"] = [pick(i, "music_director", "") for i in ids]
    out["cast_names"] = [pick(i, "cast_names", []) for i in ids]
    out["keyword_names"] = [pick(i, "keyword_names", []) for i in ids]
    for raw in ("Director", "Music_Composer"):
        if raw in out.columns:
            out[raw] = out["director"] if raw == "Director" else out["music_director"]
    if "Cast_list" in out.columns:
        out["Cast_list"] = out["cast_names"].apply(str)
    out["combined_features"] = out.apply(rebuild_features, axis=1)
    return out


def parse_args() -> argparse.Namespace:
    """Parse CLI arguments.

    Returns:
        argparse.Namespace: Parsed arguments.
    """
    p = argparse.ArgumentParser(description="Fix credits with the TMDB API.")
    p.add_argument("--movies-parquet", default=str(MERGED_MOVIES_PARQUET))
    p.add_argument("--top", type=int, default=20000,
                   help="Fetch the N most-voted matched movies (0 = all).")
    p.add_argument("--rate", type=float, default=30.0, help="Requests per second.")
    p.add_argument("--workers", type=int, default=16)
    return p.parse_args()


def main() -> int:
    """Fetch credits, rewrite the parquet and report coverage.

    Returns:
        int: Exit code (0 success, 1 failure).
    """
    args = parse_args()
    api_key = os.getenv("TMDB_API_KEY")
    if not api_key:
        logger.error("TMDB_API_KEY is not set (add it as a Kaggle Secret / env var).")
        return 1
    path = Path(args.movies_parquet)
    if not path.exists():
        logger.error("%s not found. Run src.preprocessing first.", path)
        return 1

    df = pd.read_parquet(path)
    ids = pd.to_numeric(df["id"], errors="coerce")
    candidates = df.assign(_id=ids).dropna(subset=["_id"])
    rank = "vote_count" if "vote_count" in candidates.columns else None
    if rank:
        candidates = candidates.sort_values(rank, ascending=False)
    wanted = [int(i) for i in candidates["_id"].drop_duplicates()]
    if args.top > 0:
        wanted = wanted[: args.top]

    cache = load_cache()
    todo = [i for i in wanted if i not in cache]
    logger.info("%d movies wanted, %d cached, %d to fetch.", len(wanted), len(wanted) - len(todo), len(todo))

    if todo:
        limiter = RateLimiter(args.rate)
        failed = 0
        with requests.Session() as session, open(CACHE_PATH, "a", encoding="utf-8") as sink, \
                ThreadPoolExecutor(max_workers=args.workers) as pool:
            futures = [pool.submit(fetch_one, session, limiter, api_key, i) for i in todo]
            for n, fut in enumerate(as_completed(futures), 1):
                rec = fut.result()
                if rec is None:
                    failed += 1
                else:
                    cache[rec["id"]] = rec
                    sink.write(json.dumps(rec) + "\n")
                if n % 1000 == 0:
                    sink.flush()
                    logger.info("Fetched %d / %d (failed: %d)", n, len(todo), failed)
        logger.info("Fetch complete. Failed after retries: %d (re-run to retry them).", failed)

    fixed = apply_credits(df, {i: cache[i] for i in wanted if i in cache})
    fixed.to_parquet(path, index=False)
    with_dir = int((fixed["director"] != "").sum())
    logger.info("Saved %s. Movies with verified director: %d / %d.", path, with_dir, len(fixed))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Overwriting /kaggle/working/cinematch-pro/scripts/enrich_credits.py


In [10]:
# CELL 8b — Fix credits from the TMDB API (needs the TMDB_API_KEY secret)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["TMDB_API_KEY"] = UserSecretsClient().get_secret("TMDB_API_KEY")
except Exception as exc:
    if not os.getenv("TMDB_API_KEY"):
        raise RuntimeError(
            "TMDB_API_KEY secret not found. Kaggle -> Add-ons -> Secrets -> add TMDB_API_KEY "
            "and tick the toggle to attach it to this notebook."
        ) from exc
!python -m scripts.enrich_credits --top 20000

2026-10-17 06:12:09 | INFO     | cinematch | 20000 movies wanted, 16741 cached, 3259 to fetch.
2026-10-17 06:12:42 | INFO     | cinematch | Fetched 1000 / 3259 (failed: 0)
2026-10-17 06:13:16 | INFO     | cinematch | Fetched 2000 / 3259 (failed: 0)
2026-10-17 06:13:49 | INFO     | cinematch | Fetched 3000 / 3259 (failed: 0)
2026-10-17 06:13:58 | INFO     | cinematch | Fetch complete. Failed after retries: 0 (re-run to retry them).
2026-10-17 06:14:00 | INFO     | cinematch | Saved /kaggle/working/cinematch-pro/data/processed/merged_movies.parquet. Movies with verified director: 20060 / 62423.


In [11]:
# CELL 8c — Verify the credits are correct now (Heat -> Michael Mann, Jumanji -> Joe Johnston)
import pandas as pd
m = pd.read_parquet("/kaggle/working/cinematch-pro/data/processed/merged_movies.parquet")
chk = m[(m["year"] == 1995) & m["clean_title"].isin(["Heat", "Jumanji", "Toy Story"])]
print(chk[["clean_title", "director", "music_director", "cast_names"]].to_string())
print("\nMovies with verified director:", int((m["director"] != "").sum()), "/", len(m))
assert (m["director"] != "").sum() > 5000, "Credits fix did not run - check CELL 8b output"

  clean_title       director     music_director                                                                                  cast_names
0   Toy Story  John Lasseter       Randy Newman           [Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn, John Ratzenberger]
1     Jumanji   Joe Johnston       James Horner  [Robin Williams, Kirsten Dunst, Bradley Pierce, Bonnie Hunt, Jonathan Hyde, Bebe Neuwirth]
5        Heat   Michael Mann  Elliot Goldenthal             [Al Pacino, Robert De Niro, Val Kilmer, Jon Voight, Tom Sizemore, Diane Venora]

Movies with verified director: 20060 / 62423


## Stage 2 · TF-IDF content filter (CPU, ~1 min)

In [12]:
# CELL 9 — Content-based filter
!python -m scripts.build_content_model

2026-10-17 06:14:02 | INFO     | cinematch | Fitting TF-IDF vectorizer on 62423 movies.
2026-10-17 06:14:12 | INFO     | cinematch | TF-IDF matrix built: shape=(62423, 20000), vocabulary_size=20000
2026-10-17 06:14:12 | INFO     | cinematch | Saved content filter artifacts to /kaggle/working/cinematch-pro/models/tfidf_matrix.pkl and /kaggle/working/cinematch-pro/models/tfidf_vectorizer.pkl
2026-10-17 06:14:12 | INFO     | cinematch | Updated model card section 'content_filter' in /kaggle/working/cinematch-pro/models/model_card.json
2026-10-17 06:14:12 | INFO     | cinematch | Content filter saved to /kaggle/working/cinematch-pro/models/tfidf_matrix.pkl


## Stage 3 · Semantic search index: SBERT → ChromaDB (uses the GPU)

In [13]:
# CELL 10 — Semantic index
!python -m scripts.ingest_descriptions --reset

2026-10-17 06:14:17 | INFO     | cinematch | Initializing ChromaDB client at /kaggle/working/cinematch-pro/chromadb_store (embedding device=cuda)
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 557kB/s]
README.md: 100%|███████████████████████████| 10.5k/10.5k [00:00<00:00, 29.3MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 258kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 2.74MB/s]

model.safetensors: downloading bytes: ██████████████▌       | 60.1MB, 3.79MB/s  
model.safetensors: downloading bytes: ██████████████████████| 85.0MB, 8.23MB/s  
model.safetensors: reconstructing file: 100%|██████| 90.9MB / 90.9MB, 8.92MB/s  
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.30MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 41.8MB/s]
tokenizer.json: 100%|█████████████████████████| 466k/466k [00:00<00:00, 119MB/s]
config.json: 100%|█████████████████████████

## Stage 4 · Collaborative filter (SVD) — CPU only
Surprise GPU use nahi karta. `--nrows` = file ke pehle N ratings = pehle kuch hazaar users ki saari ratings (ye RAM ko control me rakhta hai).

In [14]:
# CELL 11 — SVD collaborative filter
!python -m scripts.build_cf_model --ratings-path "{RATINGS}" --nrows {N_RATINGS}

2026-10-17 06:16:50 | INFO     | cinematch | Loading MovieLens ratings from /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/ratings.csv (nrows=5000000)
2026-10-17 06:16:51 | INFO     | cinematch | Loaded 5000000 ratings from 32564 unique users.
2026-10-17 06:16:52 | INFO     | cinematch | Cleaned ratings: 5000000 -> 5000000 rows (0 duplicates/invalid removed).
2026-10-17 06:17:05 | INFO     | cinematch | Training SVD: n_factors=100, n_epochs=20, lr_all=0.0050 on 4000000 ratings.
2026-10-17 06:18:03 | INFO     | cinematch | SVD training complete. Test RMSE=0.8005 (target <0.87)
2026-10-17 06:18:09 | INFO     | cinematch | Saved SVD model to /kaggle/working/cinematch-pro/models/svd_model.pkl
2026-10-17 06:18:09 | INFO     | cinematch | Updated model card section 'collaborative_filter' in /kaggle/working/cinematch-pro/models/model_card.json
2026-10-17 06:18:09 | INFO     | cinematch | Collaborative filter trained. RMSE=0.8005. Saved to /kaggle/working/cinematch-pro/models/svd_model.pkl

In [15]:
# CELL: patch visual_search.py
import re
p = "/kaggle/working/cinematch-pro/src/visual_search.py"
s = open(p).read()

helper = '''
def _to_feature_tensor(output):
    """Return a plain tensor from CLIP feature output (new/old transformers)."""
    if isinstance(output, torch.Tensor):
        return output
    pooled = getattr(output, "pooler_output", None)
    return pooled if pooled is not None else output[0]

'''
if "_to_feature_tensor" not in s:
    s = s.replace("self.model.get_image_features(**inputs)", "_to_feature_tensor(self.model.get_image_features(**inputs))")
    s = s.replace("self.model.get_text_features(**inputs)", "_to_feature_tensor(self.model.get_text_features(**inputs))")
    i = s.index("\nclass ")
    s = s[:i] + "\n" + helper + s[i:]
    open(p, "w").write(s)
print("patched")

patched


## Stage 5 · Poster visual search: CLIP + FAISS
`MovieGenre.csv` me sirf poster **URLs** hain. Script unhe `links.csv` se `movieId` se jodta hai, download karta hai (resumable), aur dead link par TMDB poster use karta hai.

In [16]:
# CELL 12 — Poster index
!python -m scripts.build_faiss_index --genre-path "{GENRE}" --links-path "{LINKS}" {POSTER_ARGS}

2026-10-17 06:18:18 | INFO     | cinematch | Loading poster metadata from /kaggle/input/datasets/neha1703/movie-genre-from-its-poster/MovieGenre.csv
2026-10-17 06:18:19 | WARNING  | cinematch | MovieGenre.csv is not valid UTF-8 — retrying with Latin-1 encoding (common for older Kaggle exports with accented characters).
2026-10-17 06:18:19 | INFO     | cinematch | Loaded metadata for 40108 posters.
2026-10-17 06:18:19 | INFO     | cinematch | Loading MovieLens links from /kaggle/input/datasets/narendrakalam/ml-25m/ml-25m/links.csv
2026-10-17 06:18:19 | INFO     | cinematch | Loaded 62423 movie id links.
2026-10-17 06:18:19 | INFO     | cinematch | Poster catalogue: 57648 movies (36262 with a primary URL, 51944 with a TMDB fallback).
2026-10-17 06:18:21 | INFO     | cinematch | Poster downloads: 1000/57648 processed, 970 ok.
2026-10-17 06:18:21 | INFO     | cinematch | Poster downloads: 2000/57648 processed, 1949 ok.
2026-10-17 06:18:21 | INFO     | cinematch | Poster downloads: 3000/576

## Stage 6 · Sentiment model: RoBERTa fine-tune (GPU, sabse lamba step)
Ye **model train** karta hai (IMDB 50k reviews se). Kisi ek movie ka sentiment dashboard baad me **local app me, runtime par** banta hai (TMDB se us movie ke reviews laake) — Kaggle par nahi.

In [17]:
# CELL 13 — Sentiment model
!python -m scripts.train_sentiment --reviews-path "{IMDB}" {SENT_ARGS}

2026-10-17 06:24:20 | INFO     | cinematch | Loading IMDB reviews dataset from /kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
2026-10-17 06:24:21 | INFO     | cinematch | Loaded 50000 reviews (25000 positive, 25000 negative).
2026-10-17 06:24:21 | INFO     | cinematch | Loading sentiment model from 'cardiffnlp/twitter-roberta-base-sentiment' onto device=cuda
config.json: 100%|█████████████████████████████| 747/747 [00:00<00:00, 3.52MB/s]
vocab.json: 100%|████████████████████████████| 899k/899k [00:00<00:00, 81.5MB/s]
merges.txt: 100%|█████████████████████████████| 456k/456k [00:00<00:00, 119MB/s]
special_tokens_map.json: 100%|██████████████████| 150/150 [00:00<00:00, 862kB/s]
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.

pytorch_model.bin: downloading bytes: █████████▎            |  212MB, 15.4MB/s  
pytorch_model.bin: downloading bytes: ███████████▊          |  267MB, 22.3MB/s  
pytorch_m

## Check · Kya sab artifacts sahi load ho rahe hain?

In [18]:
# CELL 14 — Verify every artifact loads
!python -m scripts.check_artifacts --deep


Artifact presence
  OK       Merged movie catalogue     data/processed/merged_movies.parquet
  OK       SVD collaborative filter   models/svd_model.pkl
  OK       TF-IDF matrix              models/tfidf_matrix.pkl
  OK       TF-IDF vectorizer          models/tfidf_vectorizer.pkl
  OK       Sentiment model config     models/roberta_sentiment/config.json
  OK       FAISS poster index         models/faiss_poster.index
  OK       FAISS poster id map        models/faiss_poster_metadata.pkl
  OK       ChromaDB store             chromadb_store/chroma.sqlite3
  OK       Model card                 models/model_card.json

Library versions (training machine vs this machine)
  OK    no version-sensitive differences

Deep load check
2026-10-17 07:25:43 | INFO     | cinematch | Loaded SVD model from /kaggle/working/cinematch-pro/models/svd_model.pkl
2026-10-17 07:25:44 | INFO     | cinematch | Loaded content filter with 62423 movies.
2026-10-17 07:25:51 | INFO     | cinematch | Loaded FAISS index w

In [19]:
# CELL 15 — Real results of this run (these are the numbers for your README)
import json
print(json.dumps({k: {m: v for m, v in section.items() if m not in ("environment",)}
                  for k, section in json.loads(MODEL_CARD_PATH.read_text()).items()}, indent=2))

{
  "collaborative_filter": {
    "rmse": 0.8005,
    "n_factors": 100,
    "n_epochs": 20,
    "lr_all": 0.005,
    "grid_search": false,
    "ratings_used": 5000000,
    "users": 32564,
    "movies": 38855,
    "updated_at": "2026-10-17T06:18:09+00:00"
  },
  "content_filter": {
    "movies": 62423,
    "vocabulary_size": 20000,
    "updated_at": "2026-10-17T06:14:12+00:00"
  },
  "semantic_search": {
    "embedding_model": "all-MiniLM-L6-v2",
    "movies_ingested": 62423,
    "collection_size": 62423,
    "updated_at": "2026-10-17T06:16:45+00:00"
  },
  "visual_search": {
    "clip_model": "openai/clip-vit-base-patch32",
    "posters_requested": 57648,
    "posters_downloaded": 56029,
    "vectors_indexed": 56029,
    "embedding_dim": 512,
    "updated_at": "2026-10-17T06:24:10+00:00"
  },
  "sentiment": {
    "accuracy": 0.9426,
    "precision": 0.9385652001585414,
    "recall": 0.9472,
    "f1": 0.9428628309775035,
    "base_model": "cardiffnlp/twitter-roberta-base-sentiment",
   

In [20]:
# CELL 16 — A few live queries so you can eyeball quality before downloading anything
from src.config import ROBERTA_MODEL_DIR
from src.content_filter import ContentBasedFilter
from src.semantic_search import SemanticSearchEngine
from src.sentiment_engine import SentimentEngine
from src.visual_search import VisualSearchEngine

title_of = dict(zip(movies["movieId"], movies["clean_title"]))

print("SEMANTIC: 'thriller with an unreliable narrator and a twist ending'")
for hit in SemanticSearchEngine().search("thriller with an unreliable narrator and a twist ending", top_k=5):
    print(f"  {hit['title']:<40} {hit['similarity_pct']}%")

content = ContentBasedFilter().load()
seed_rows = movies[movies["clean_title"].str.contains("Inception", na=False)]
seed = int(seed_rows["movieId"].iloc[0]) if len(seed_rows) else int(movies["movieId"].iloc[0])
print(f"\nCONTENT: more like '{title_of[seed]}'")
for movie_id, score in content.similar_movies(seed, top_k=5):
    print(f"  {title_of.get(movie_id, movie_id):<40} {score:.2f}")

print("\nVISUAL: 'dark moody thriller poster'")
visual = VisualSearchEngine().load()
for movie_id, pct in visual.search_by_text("dark moody thriller poster", top_k=5):
    print(f"  {title_of.get(movie_id, movie_id):<40} {pct}%")

print("\nSENTIMENT")
engine = SentimentEngine(model_path=ROBERTA_MODEL_DIR)
for text in ["A moving, beautifully acted film.", "Dull, predictable and far too long."]:
    result = engine.predict(text)
    print(f"  {result.label:<9} {result.confidence:.2f}  <- {text}")

SEMANTIC: 'thriller with an unreliable narrator and a twist ending'
2026-10-17 07:26:00 | INFO     | cinematch | Initializing ChromaDB client at /kaggle/working/cinematch-pro/chromadb_store (embedding device=cuda)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Labyrinth                                85.6%
  Venetian Bird                            85.6%
  The Mysterious Mr. Wong                  85.6%
  Ittefaq: It happened one night           85.6%
  Devil You Know                           85.6%
2026-10-17 07:26:02 | INFO     | cinematch | Loaded content filter with 62423 movies.

CONTENT: more like 'Inception'
  Carnival of Souls                        0.19
  Comedy Central Roast of Bruce Willis     0.17
  Don Jon                                  0.15
  Freud: The Secret Passion                0.15
  G.I. Joe: The Rise of Cobra              0.14

VISUAL: 'dark moody thriller poster'
2026-10-17 07:26:02 | INFO     | cinematch | Loading CLIP model 'openai/clip-vit-base-patch32' onto device=cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

2026-10-17 07:26:03 | INFO     | cinematch | Loaded FAISS index with 56029 vectors from /kaggle/working/cinematch-pro/models/faiss_poster.index
  Darker than Night                        32.3%
  Alone in the Dark                        32.1%
  Beneath the Dark                         32.0%
  Chained                                  32.0%
  The Witch in the Window                  31.4%

SENTIMENT
2026-10-17 07:26:03 | INFO     | cinematch | Loading sentiment model from '/kaggle/working/cinematch-pro/models/roberta_sentiment' onto device=cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  positive  1.00  <- A moving, beautifully acted film.
  negative  1.00  <- Dull, predictable and far too long.


## Download · Outputs ko ek zip me pack karo

In [21]:
# CELL 17 — Bundle everything you need locally into ONE zip (this is the file you download)
import shutil, zipfile

CLEAN_OUTPUT = True   # True: afterwards delete the working copy so the Output panel shows ONLY the zip

shutil.rmtree(PROJECT / "data" / "posters", ignore_errors=True)   # ~1 GB of cached JPEGs, not needed locally
BUNDLE = WORK / "cinematch_artifacts.zip"
include = [PROJECT / "models", PROJECT / "chromadb_store", PROJECT / "data" / "processed" / "merged_movies.parquet"]

with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for root in include:
        files = [root] if root.is_file() else [p for p in root.rglob("*") if p.is_file()]
        for file in files:
            archive.write(file, file.relative_to(PROJECT))

print(f"{BUNDLE}  {BUNDLE.stat().st_size / 1e6:,.0f} MB")
with zipfile.ZipFile(BUNDLE) as archive:
    for info in sorted(archive.infolist(), key=lambda i: -i.file_size)[:12]:
        print(f"  {info.file_size / 1e6:9,.1f} MB  {info.filename}")

if CLEAN_OUTPUT:
    os.chdir(WORK)
    shutil.rmtree(PROJECT, ignore_errors=True)
    print("\nWorking copy removed. Output now contains only cinematch_artifacts.zip "
          "(re-run CELL 5 onward if you need the project again).")

/kaggle/working/cinematch_artifacts.zip  897 MB
      498.6 MB  models/roberta_sentiment/model.safetensors
      168.0 MB  models/svd_model.pkl
      163.0 MB  chromadb_store/chroma.sqlite3
      114.7 MB  models/faiss_poster.index
      103.0 MB  chromadb_store/1d06d154-e5b8-4fe4-8d66-517725da4d37/data_level0.bin
       40.4 MB  data/processed/merged_movies.parquet
       23.6 MB  models/tfidf_matrix.pkl
        3.6 MB  models/roberta_sentiment/tokenizer.json
        1.9 MB  chromadb_store/1d06d154-e5b8-4fe4-8d66-517725da4d37/index_metadata.pickle
        0.8 MB  models/tfidf_vectorizer.pkl
        0.5 MB  chromadb_store/1d06d154-e5b8-4fe4-8d66-517725da4d37/link_lists.bin
        0.4 MB  models/faiss_poster_metadata.pkl

Working copy removed. Output now contains only cinematch_artifacts.zip (re-run CELL 5 onward if you need the project again).


In [22]:
import os, sys, zipfile
P = "/kaggle/working/cinematch-pro"
zipfile.ZipFile("/kaggle/working/cinematch_artifacts.zip").extractall(P)  # models + chromadb + parquet wapas
os.chdir(P); sys.path.insert(0, P)

import pandas as pd
m = pd.read_parquet("data/processed/merged_movies.parquet")
print("--- Inception wale rows ---")
print(m[m["clean_title"].str.contains("Inception", na=False)][["movieId", "clean_title", "year"]].head(10))

from src.semantic_search import SemanticSearchEngine
eng = SemanticSearchEngine()
q = "thriller with an unreliable narrator and a twist ending"
r = eng.collection.query(query_texts=[q], n_results=10, include=["distances", "documents", "metadatas"])
print("\n--- Semantic: raw distances ---")
for d, doc, meta in zip(r["distances"][0], r["documents"][0], r["metadatas"][0]):
    print(round(d, 4), "|", meta["title"], "|", doc[:90].replace("\n", " "))

--- Inception wale rows ---
       movieId clean_title  year
14937    79132   Inception  2010
2026-10-17 07:46:49 | INFO     | cinematch | Initializing ChromaDB client at /kaggle/working/cinematch-pro/chromadb_store (embedding device=cuda)

--- Semantic: raw distances ---
0.2876 | Labyrinth | Thriller Mystery
0.2877 | Jennifer 8 | Mystery Thriller
0.2877 | Keane | Mystery Thriller
0.2877 | Vita frun | Mystery Thriller
0.2877 | Psychophony | Mystery Thriller
0.2877 | Venetian Bird | Mystery Thriller
0.2877 | Q Planes | Mystery Thriller
0.2877 | The Mysterious Mr. Wong | Mystery Thriller
0.2877 | Ittefaq: It happened one night | Mystery Thriller
0.2877 | Devil You Know | Mystery Thriller


### Zip ko download karna
- **Interactive session me:** right panel → **Output** (ya `/kaggle/working`) → `cinematch_artifacts.zip` ke saamne **⋮ → Download**.
- **Save & Run All (Commit) ke baad:** notebook page → **Versions** → apna version kholo → **Output** tab → zip → **Download**.

### Local machine par ab kya karna hai
1. Zip ko **project root me extract** karo (`models/`, `chromadb_store/`, `data/processed/` tumhare folders me merge ho jayenge).
2. Conda env me: `conda install -c conda-forge scikit-surprise` phir `pip install -r requirements.txt`.
3. `python -m scripts.check_artifacts --deep`
4. Mismatch/failed-load aaye to ye do sasti cheezein local par dobara banao:
   `python -m scripts.build_content_model` aur `python -m scripts.ingest_descriptions --reset`
5. API: `python -m scripts.run_api` · App: `python -m scripts.run_app`